# مشروع توقع أسعار العقارات باستخدام تعلم الآلة 🏠📊
مرحباً بك في هذا المشروع! الهدف هنا هو بناء وتدريب نموذج ذكاء اصطناعي قادر على توقع أسعار العقارات بناءً على مواصفاتها الأساسية (المساحة، عدد الغرف، وعدد دورات المياه).

---
## 1. استيراد المكتبات الأساسية
في هذه الخطوة، نقوم باستيراد المكتبات اللازمة للتعامل مع البيانات وبناء النموذج.

In [20]:
import re
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score


## 2. تحميل البيانات وتجهيز النموذج
هنا نقوم بالخطوات التالية:
1. قراءة ملف البيانات `Aqar_data.csv` وتحديد الأعمدة الرقمية المهمة فقط.
2. فصل المتغيرات المستقلة (المميزات `x`) عن المتغير التابع (الهدف `y` وهو السعر).
3. تعريف نموذج **Random Forest Regressor** مع ضبط المعلمات (Hyperparameters) للحد من مشكلة الـ *Overfitting*.

In [21]:
df = pd.read_csv("Aqar_data.csv")
df['neighborhood'] = df['location'].str.replace('حي', '').str.replace('- الرياض', '').str.strip()


## 3. تقسيم البيانات وتقييم أداء النموذج
نقوم بتقسيم البيانات إلى مجموعتي تدريب واختبار بنسبة (80:20)، ثم ندرب النموذج ونحسب دقة التوقع ($R^2\ Score$) للتأكد من توازن الأداء وعدم وجود حفظ صم للبيانات (*Overfitting*).

In [22]:
# 1. تجهيز النصوص واستخراج الميزات الفاخرة المتقدمة
df['neighborhood'] = df['location'].str.replace('حي', '').str.replace('- الرياض', '').str.strip()
df['listTitle'] = df['listTitle'].fillna('')
df['details'] = df['details'].fillna('')
full_text = df['details'] + ' ' + df['listTitle']

arabic_digits = {'٠':'0', '١':'1', '٢':'2', '٣':'3', '٤':'4', '٥':'5', '٦':'6', '٧':'7', '٨':'8', '٩':'9'}
def normalize_text(s):
    for k, v in arabic_digits.items():
        s = s.replace(k, v)
    return s

norm_text = full_text.apply(normalize_text)

# الميزات الأساسية والنوع
df['is_villa'] = (df['listTitle'].str.contains('فيلا', na=False) | df['details'].str.contains('فيلا', na=False)).astype(int)
df['is_apartment'] = (df['listTitle'].str.contains('شقة|شقه', regex=True, na=False) | df['details'].str.contains('شقة|شقه', regex=True, na=False)).astype(int)
df['has_driver_room'] = df['details'].str.contains('سائق|سواق', regex=True, na=False).astype(int)
df['has_maid_room'] = df['details'].str.contains('خادمة|شغالة', regex=True, na=False).astype(int)
df['is_new'] = df['details'].str.contains('جديد|جديدة', regex=True, na=False).astype(int)

df['has_elevator'] = norm_text.str.contains('مصعد|اصنصير', regex=True, na=False).astype(int)
df['has_pool'] = norm_text.str.contains('مسبح', na=False).astype(int)
df['has_annex'] = norm_text.str.contains('ملحق', na=False).astype(int)
df['has_garage'] = norm_text.str.contains('مدخل سيارة|مدخل سياره|كراج', regex=True, na=False).astype(int)
df['has_central_ac'] = norm_text.str.contains('تكييف مخفي|تكيف مخفي|مركزي|سبلت مخفي', regex=True, na=False).astype(int)
df['has_duplex_stairs'] = norm_text.str.contains('درج صالة|درج صاله|درج داخلي|دوبلكس', regex=True, na=False).astype(int)

def extract_street_width(text):
    m = re.search(r'(?:شارع|عرض)\s*(\d{2,3})', str(text))
    if m:
        w = int(m.group(1))
        if 8 <= w <= 100: return float(w)
    return 15.0
df['street_width'] = norm_text.apply(extract_street_width)

def extract_extra_apartments(text):
    text = str(text)
    if 'شقتين' in text or 'مع شقتين' in text or 'وشقتين' in text: return 2
    elif 'شقة' in text or 'مع شقة' in text or 'وشقة' in text: return 1
    return 0
df['extra_apartments'] = norm_text.apply(extract_extra_apartments)

# 2. ميزات الفخامة الحديثة (Modern Luxury Signals)
# أ. غرف الماستر
def extract_master_bedrooms(t):
    m = re.search(r'(\d+)\s*(?:غرف(?:ة)?\s*(?:نوم\s*)?)?(?:ماستر|أجنحة|اجنحه)', t)
    if m:
        c = int(m.group(1))
        if 1 <= c <= 8: return float(c)
    if 'ماستر' in t or 'جناح رئيسي' in t: return 1.0
    return 0.0
df['master_bedrooms'] = norm_text.apply(extract_master_bedrooms)

# ب. الصالات والمجالس
def extract_living_rooms(t):
    count = 0
    m_majlis = re.search(r'(\d+)\s*مجالس', t)
    if m_majlis: count += int(m_majlis.group(1))
    elif 'مجلسين' in t: count += 2
    elif 'مجلس' in t: count += 1
    m_sala = re.search(r'(\d+)\s*صالات', t)
    if m_sala: count += int(m_sala.group(1))
    elif 'صالتين' in t: count += 2
    elif 'صالة' in t or 'صاله' in t: count += 1
    return float(min(count, 8))
df['living_rooms'] = norm_text.apply(extract_living_rooms)

# ج. الواجهة والاتجاه
def extract_facade(t):
    if re.search(r'شارعين|زاوية|زاويه|رأس بلك|راس بلك|ثلاث شوارع', t): return 'corner'
    elif re.search(r'شمالي|شماليه|شمالية', t): return 'north'
    elif re.search(r'شرقي|شرقيه|شرقية', t): return 'east'
    elif re.search(r'جنوبي|جنوبيه|جنوبية', t): return 'south'
    elif re.search(r'غربي|غربيه|غربية', t): return 'west'
    return 'unknown'
df['facade'] = norm_text.apply(extract_facade)

# د. علامات الفخامة العصرية
df['has_double_height'] = norm_text.str.contains('دبل هايت|دبلهايت|اسقف عاليه|أسقف عالية|سقف عالي|ارتفاع عالي|سقف مرتفع', regex=True).astype(int)
df['elevator_installed'] = norm_text.str.contains('مصعد راكب|راكب مصعد|اصنصير|مصعد شغال|المصعد راكب', regex=True).astype(int)
df['elevator_ready'] = norm_text.str.contains('تأسيس مصعد|تاسيس مصعد|مؤسس مصعد|مكان مصعد', regex=True).astype(int)
df['has_custom_built'] = norm_text.str.contains('بناء شخصي|إشراف هندسي|اشراف هندسي|كود سعودي|تأمين ملاذ|شهادة إتمام', regex=True).astype(int)
df['has_spacious_yard'] = norm_text.str.contains('حوش|فناء|حديقة|ارتداد واسع|مسطحات خضراء|جلسة خارجية|جلسه خارجيه', regex=True).astype(int)
df['is_modern'] = norm_text.str.contains('مودرن|عصري|تصميم حديث|تصميم عصري|فيلا مودرن', regex=True).astype(int)

# ترميز الواجهة One-Hot
facade_dummies = pd.get_dummies(df['facade'], prefix='facade', drop_first=True, dtype=int)
df = pd.concat([df, facade_dummies], axis=1)

# 3. تنظيف البيانات بحدود نطاق سكني واقعي لمنع التسريب
clean_df = df[
    (df['price'] >= 150000) &
    (df['price'] <= 7000000) &
    (df['size'] >= 100) &
    (df['size'] <= 1200)
].copy()

# 4. تحويل السعر اللوغاريتمي وتقسيم البيانات
drop_cols = ['price', 'listTitle', 'details', 'location', 'time', 'facade']
X = clean_df.drop(columns=drop_cols)
y = np.log1p(clean_df['price'])
y_orig = clean_df['price']

X_train, X_test, y_train, y_test, y_orig_train, y_orig_test = train_test_split(
    X, y, y_orig, test_size=0.2, random_state=42
)
X_train = X_train.copy()
X_test = X_test.copy()

# 5. Out-of-Fold Target Encoding للأحياء بدون تسريب
cv_kf = KFold(n_splits=5, shuffle=True, random_state=42)
te_enc = TargetEncoder(smooth='auto', cv=5)
X_train['neighborhood_encoded'] = te_enc.fit_transform(X_train[['neighborhood']], y_train)
X_test['neighborhood_encoded'] = te_enc.transform(X_test[['neighborhood']])

feature_cols = [c for c in X_train.columns if c != 'neighborhood']

# 6. تدريب النموذج مع تقنين صارم
model = GradientBoostingRegressor(
    n_estimators=160,
    max_depth=4,
    min_samples_leaf=12,
    min_samples_split=24,
    subsample=0.85,
    max_features=0.85,
    learning_rate=0.05,
    random_state=42
)
model.fit(X_train[feature_cols], y_train)

# حساب الأداء
train_score = model.score(X_train[feature_cols], y_train)
test_score = model.score(X_test[feature_cols], y_test)
gap = (train_score - test_score) * 100

pred_sar = np.expm1(model.predict(X_test[feature_cols]))
mae_sar = mean_absolute_error(y_orig_test, pred_sar)
mape = mean_absolute_percentage_error(y_orig_test, pred_sar) * 100

print(f'دقة التدريب (R²): {train_score:.4f}')
print(f'دقة الاختبار (R²): {test_score:.4f}')
print(f'فجوة التعميم (Overfitting Gap): {gap:.2f}%')
print(f'متوسط الخطأ المطلق: {mae_sar:,.0f} ريال')
print(f'متوسط نسبة الخطأ (MAPE): {mape:.2f}%')



دقة التدريب (R²): 0.8675
دقة الاختبار (R²): 0.8684
فجوة التعميم (Overfitting Gap): -0.09%
متوسط الخطأ المطلق: 169,322 ريال
متوسط نسبة الخطأ (MAPE): 10.60%


## 4. نظام التوقع التفاعلي مع مؤشرات عقار 2026 المحدثة 📈🏠
في هذه الخطوة، نقوم بتدريب النموذج على كامل البيانات المتاحة مع ربطه ديناميكياً بمؤشرات منصة **عقار** السكنية المعتمدة لعام 2026 (عبر `AqarBenchmarkMapper`) بدون أي ثوابت تضخم ثابتة.
يقوم النظام بحساب السعر التاريخي للفيلا وفق مواصفاتها الهندسية الفاخرة، ثم معايرتها آلياً بمعدل النمو الرأسمالي الحقيقي لكل حي وفئة مساحة.

In [26]:
import os
import sys
sys.path.append(os.getcwd())
from aqar_benchmark_mapper import AqarBenchmarkMapper

# 1. تهيئة محول المؤشرات السوقية لعام 2026
benchmark_mapper = AqarBenchmarkMapper("aqar_benchmarks_2026.json")

# 2. تدريب النموذج على كامل البيانات بعد المعالجة
full_te_enc = TargetEncoder(smooth='auto', cv=5)
clean_df_encoded = clean_df.copy()
clean_df_encoded['neighborhood_encoded'] = full_te_enc.fit_transform(clean_df[['neighborhood']], y)

X_full = clean_df_encoded[feature_cols]
y_full = y

final_model = GradientBoostingRegressor(
    n_estimators=160,
    max_depth=4,
    min_samples_leaf=12,
    min_samples_split=24,
    subsample=0.85,
    max_features=0.85,
    learning_rate=0.05,
    random_state=42
)
final_model.fit(X_full, y_full)

print("---------------- تم تدريب النموذج بنجاح ----------------")

# 3. إدخال بيانات العقار التفاعلية الشاملة
user_size = float(input("أدخل المساحة (م²): ") or 200.0)
user_beds = float(input("أدخل عدد غرف النوم الكلي: ") or 5.0)
user_baths = float(input("أدخل عدد دورات المياه: ") or 5.0)
user_nh = input("أدخل اسم الحي (مثل النرجس، الورود، الملقا): ").strip() or "النرجس"

user_masters = float(input("عدد غرف النوم الماستر: ") or 4.0)
user_living = float(input("عدد الصالات والمجالس الكلي: ") or 2.0)
user_facade = input("الواجهة (شمالية / شرقية / غربية / جنوبية / زاوية): ").strip() or "شمالية"

# ميزات الفخامة والتأسيس
has_double_height = int(input("هل يوجد سقف عالي / دبل هايت؟ (1 نعم / 0 لا): ") or 1)
elevator_installed = int(input("هل المصعد راكب؟ (1 نعم / 0 لا): ") or 1)
elevator_ready = int(input("هل المصعد مؤسس (بدون كابينة)؟ (1 نعم / 0 لا): ") or 0)
has_central_ac = int(input("هل يوجد تكييف مخفي / مركزي؟ (1 نعم / 0 لا): ") or 1)
has_custom_built = int(input("هل البناء شخصي / كود سعودي / إشراف هندسي؟ (1 نعم / 0 لا): ") or 0)

# ميزات الاستثمار والمرافق
extra_apts = int(input("عدد الشقق الإضافية المسروقة/المستقلة (0 أو 1 أو 2): ") or 0)
has_driver = int(input("هل يوجد غرفة سائق؟ (1 نعم / 0 لا): ") or 1)
has_maid = int(input("هل يوجد غرفة خادمة؟ (1 نعم / 0 لا): ") or 1)
has_annex = int(input("هل يوجد ملحق خارجي؟ (1 نعم / 0 لا): ") or 0)
has_garage = int(input("هل يوجد مدخل سيارة / كراج؟ (1 نعم / 0 لا): ") or 1)
street_width = float(input("عرض الشارع بالمتر (مثلاً 12 أو 15 أو 20): ") or 20.0)

# تجهيز ميزات الواجهة
facade_map = {
    'facade_corner': 1 if ('زاوية' in user_facade or 'شارعين' in user_facade) else 0,
    'facade_east': 1 if 'شرق' in user_facade else 0,
    'facade_north': 1 if 'شمال' in user_facade else 0,
    'facade_south': 1 if 'جنوب' in user_facade else 0,
    'facade_unknown': 1 if not user_facade else 0,
    'facade_west': 1 if 'غرب' in user_facade else 0
}

# تشفير الحي
clean_nh = benchmark_mapper.clean_district(user_nh)
user_df_nh = pd.DataFrame([{'neighborhood': clean_nh}])
user_encoded_nh = float(full_te_enc.transform(user_df_nh[['neighborhood']])[0, 0])

# بناء سجل المدخلات
input_dict = {
    'size': user_size, 
    'bedrooms': user_beds, 
    'bathrooms': user_baths,
    'is_villa': 1, 
    'is_apartment': 0, 
    'has_driver_room': has_driver, 
    'has_maid_room': has_maid, 
    'is_new': 1,
    'has_elevator': elevator_installed, 
    'has_pool': 0, 
    'has_annex': has_annex, 
    'has_garage': has_garage,
    'has_central_ac': has_central_ac, 
    'has_duplex_stairs': 0, 
    'street_width': street_width, 
    'extra_apartments': extra_apts,
    'master_bedrooms': user_masters, 
    'living_rooms': user_living,
    'has_double_height': has_double_height, 
    'elevator_installed': elevator_installed, 
    'elevator_ready': elevator_ready,
    'has_custom_built': has_custom_built, 
    'has_spacious_yard': 1, 
    'is_modern': 1,
    'neighborhood_encoded': user_encoded_nh
}
input_dict.update(facade_map)

user_input = pd.DataFrame([input_dict])[feature_cols]

# 4. التوقع والمعايرة باستخدام مؤشرات عقار المعتمدة
predict_log = final_model.predict(user_input)
predict_price_hist = float(np.expm1(predict_log)[0])

predict_price_2026, bm = benchmark_mapper.calibrate(predict_price_hist, user_nh, user_size)

print("-" * 55)
print(f"الحي: {user_nh} | فئة المساحة: {bm['size_bucket']}")
print(f"مؤشر النمو الرأسمالي (عقار 2026/2020): {bm['growth_ratio']:.4f}x (تقدير وسيط: {bm['is_imputed']})")
print(f"متوسط الحي المرجعي في مؤشرات عقار (2026): {bm['avg_price_2026']:,.0f} ريال")
print(f"التقييم التاريخي للنموذج (2020-2022): {predict_price_hist:,.2f} ريال")
print(f"التقييم الفعلي المعاير لسوق 2026: {predict_price_2026:,.2f} ريال")
print("-" * 55)



---------------- تم تدريب النموذج بنجاح ----------------
-------------------------------------------------------
الحي: النرجس | فئة المساحة: 200-275 m²
مؤشر النمو الرأسمالي (عقار 2026/2020): 2.1165x (تقدير وسيط: False)
متوسط الحي المرجعي في مؤشرات عقار (2026): 2,982,070 ريال
التقييم التاريخي للنموذج (2020-2022): 1,446,262.62 ريال
التقييم الفعلي المعاير لسوق 2026: 3,061,086.34 ريال
-------------------------------------------------------
